# [8.3] ACDC and Circuit Metrics - Solutions

This notebook validates the helper contracts, inspects the committed CUDA signature result, and then reruns the live TransformerLens circuit-metric preflight through `solutions.py`.

<details>
<summary>Expected output</summary>

All local tests should pass. The signature result should show patch scores `[0, 0, 0, 0, 0, 1]`, kept circuit `['position_5']`, preserved fraction `1.0`, minimality damage about `6.2771`, omitted-node gain `0.0`, random-baseline margin about `6.2771`, three held-out recoveries at `1.0`, and a live CUDA run on `torch 2.12.1+cu132` with CUDA runtime `13.2`.

</details>

<details>
<summary>Help - why rerun live CUDA?</summary>

The committed report is the review artifact, but the live cell proves the current `uv` environment, GPU, TransformerLens loader, tokenizer revision, patching hook, and metric battery still work together.

</details>


In [ ]:
import json
import sys
from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter8_automated_circuits"
section = "part3_acdc_circuit_metrics"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_acdc_circuit_metrics.tests as tests
import part3_acdc_circuit_metrics.utils as utils

GT_TIER = "GT-1"
EXERCISE_ID = "8_3_acdc_and_circuit_metrics"
DIFFICULTY = 4
IMPORTANCE = 4
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True

from part3_acdc_circuit_metrics import solutions


## Unit Contracts

The tests catch metric direction, degenerate patch scores, invalid pruning reports, invalid metric thresholds, broken OOD grouping, and weak exact-vs-approximate method comparisons before the live model path.

<details>
<summary>Help - why not only test the CUDA report?</summary>

A single CUDA boolean is too coarse for debugging. If a circuit claim fails, you need to know whether the issue is the metric readout, pruning, faithfulness, minimality, completeness, random controls, OOD checks, or method comparison.

</details>


In [ ]:
tests.test_position_patching_helpers_score_recovery(
    solutions.answer_logit_diff,
    solutions.activation_patching_sweep,
)
tests.test_position_patching_helpers_reject_degenerate_inputs(
    solutions.answer_logit_diff,
    solutions.activation_patching_sweep,
)
tests.test_acdc_pruning_report_keeps_threshold_edges(solutions.acdc_pruning_report)
tests.test_acdc_pruning_report_rejects_bad_scores_or_names(solutions.acdc_pruning_report)
tests.test_faithfulness_report_normalizes_clean_corrupt_gap(
    solutions.circuit_faithfulness_report,
)
tests.test_minimality_and_completeness_reports_distinguish_failure_modes(
    solutions.circuit_minimality_report,
    solutions.circuit_completeness_report,
)
tests.test_random_circuit_baseline_report_requires_margin(
    solutions.random_circuit_baseline_report,
)
tests.test_circuit_metric_reports_reject_invalid_thresholds(
    solutions.circuit_faithfulness_report,
    solutions.circuit_minimality_report,
    solutions.circuit_completeness_report,
    solutions.random_circuit_baseline_report,
)
tests.test_ood_template_report_tracks_worst_template(solutions.ood_template_report)
tests.test_ood_template_report_rejects_degenerate_inputs(solutions.ood_template_report)
tests.test_circuit_method_comparison_report_matches_exact_patching(
    solutions.circuit_method_comparison_report,
)
tests.test_circuit_method_comparison_report_rejects_bad_inputs(
    solutions.circuit_method_comparison_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## CPU Contract

Before the live model, the smoke report should already have the section shape: pruning, faithfulness, minimality, completeness, random baseline, OOD templates, and exact-vs-approximate method comparison.

<details>
<summary>Expected output</summary>

`num_kept` should be `2`, preserved fraction should be `0.8`, minimality damage should be `1.7`, omitted-node gain should be `0.15`, random margin should be `1.4`, OOD should pass, and method comparison should pass for the single good method in the smoke fixture.

</details>

<details>
<summary>Common bug</summary>

Do not return dataclass objects directly from the smoke report. The verification report path expects JSON-like dictionaries.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["acdc"]["num_kept"] == 2
assert contract["faithfulness"]["preserved_fraction"] == 0.8
assert contract["minimality"]["metric_damage"] == 1.7000000000000002
assert abs(contract["completeness"]["omitted_node_gain"] - 0.15) < 1e-6
assert abs(contract["random_baseline"]["margin"] - 1.4) < 1e-6
assert contract["ood"]["passes_ood"]
assert contract["method_comparison"]["passes_comparison"]
utils.print_report(
    "CPU circuit-metric contract",
    {
        "kept_edges": contract["acdc"]["kept_edges"],
        "preserved_fraction": contract["faithfulness"]["preserved_fraction"],
        "minimality_damage": round(contract["minimality"]["metric_damage"], 4),
        "omitted_node_gain": round(contract["completeness"]["omitted_node_gain"], 4),
        "random_margin": round(contract["random_baseline"]["margin"], 4),
        "ood_worst_accuracy": contract["ood"]["worst_template_accuracy"],
        "method_topk_overlap": contract["method_comparison"]["topk_overlap"],
    },
)


## Signature Result

Now inspect the accepted CUDA report. The important result is the metric battery, not only the final boolean.

<details>
<summary>Interpreting the signature result</summary>

Exact patching selects only final residual position `5`. The thresholded circuit keeps `position_5`. That circuit preserves the metric, removing it is damaging, adding the best omitted position does not help, wrong-position controls fail, and held-out prompt templates still localize the final position. This is a metrics preflight, not full ACDC discovery.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"] and report["tests_passed"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
assert gpu["best_position"] == gpu["target_position"] == 5
assert gpu["kept_edges"] == ["position_5"]
assert gpu["passes_faithfulness"]
assert gpu["passes_minimality"]
assert gpu["passes_completeness"]
assert gpu["circuit_beats_random"]
assert gpu["passes_ood"]
assert gpu["template_count"] == 3
assert gpu["peak_vram_gb"] <= 24.0

positions = list(range(gpu["sequence_length"]))
fig, (ax_scores, ax_checks) = plt.subplots(
    1,
    2,
    figsize=(9, 3.3),
    gridspec_kw={"width_ratios": [1.7, 1.3]},
)
ax_scores.bar(positions, gpu["patch_scores_by_position"], color="#2563eb")
ax_scores.set_title("Exact patch recovery")
ax_scores.set_xlabel("sequence position")
ax_scores.set_ylabel("recovered fraction")
ax_scores.set_xticks(positions)
ax_scores.set_ylim(0, 1.1)

check_names = ["faith", "minimal", "complete", "random", "heldout"]
check_values = [1, 1, 1, 1, 1]
ax_checks.barh(check_names, check_values, color="#10b981")
ax_checks.set_xlim(0, 1.1)
ax_checks.set_title("Circuit metric checks")
ax_checks.set_xlabel("pass")
fig.tight_layout()
plt.show()

utils.print_report(
    "Committed CUDA signature result",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "patch_scores": gpu["patch_scores_by_position"],
        "kept_edges": gpu["kept_edges"],
        "preserved_fraction": gpu["preserved_fraction"],
        "minimality_damage": round(gpu["minimality_metric_damage"], 4),
        "top_omitted_position": gpu["top_omitted_position"],
        "omitted_node_gain": gpu["omitted_node_gain"],
        "random_margin": round(gpu["random_baseline_margin"], 4),
        "template_recoveries": gpu["template_recoveries"],
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)


## Live CUDA Path

The report above is committed evidence. This cell reruns the live CUDA path on the current machine through `solutions.py`.

<details>
<summary>Expected output</summary>

`preflight_passed` should be `True`, peak VRAM should stay below the 24GB budget, the kept circuit should be `['position_5']`, all metric checks should pass, and there should be three held-out template recoveries.

</details>


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_full_experiment(max_vram_gb=max_vram_gb)


live_gpu = run_full_experiment(max_vram_gb=24.0)
assert live_gpu["preflight_passed"]
assert live_gpu["patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
assert live_gpu["best_position"] == live_gpu["target_position"] == 5
assert live_gpu["kept_edges"] == ["position_5"]
assert live_gpu["passes_faithfulness"]
assert live_gpu["passes_minimality"]
assert live_gpu["passes_completeness"]
assert live_gpu["circuit_beats_random"]
assert live_gpu["passes_ood"]
assert live_gpu["template_count"] == 3
assert live_gpu["peak_vram_gb"] <= 24.0
utils.print_report(
    "Live CUDA ACDC/circuit-metric preflight",
    {
        "torch": live_gpu["torch_version"],
        "cuda": live_gpu["cuda_version"],
        "device": live_gpu["device"],
        "patch_scores": live_gpu["patch_scores_by_position"],
        "kept_edges": live_gpu["kept_edges"],
        "metric_checks": {
            "faithfulness": live_gpu["passes_faithfulness"],
            "minimality": live_gpu["passes_minimality"],
            "completeness": live_gpu["passes_completeness"],
            "random": live_gpu["circuit_beats_random"],
            "heldout": live_gpu["passes_ood"],
        },
        "template_recoveries": live_gpu["template_recoveries"],
        "peak_vram_gb": round(live_gpu["peak_vram_gb"], 4),
    },
)


## Limitations

This is a GT-1 position-circuit metric preflight on one pinned `gelu-1l` hook, one primary prompt pair, and three held-out safe prompt-template pairs. It is not full ACDC, not IOI replication, not greater-than circuit discovery, and not evidence for a layer/head/feature/path circuit.

## Further Research

Implement real edge-level ACDC, compare thresholded exact-patch circuits to EAP/EAP-IG across several granularities, add threshold-sensitivity curves, and reproduce a small IOI or greater-than fragment with real path-patching semantics.
